In [1]:
import pandas as pd
from datetime import datetime
import duckdb

Когортный анализ Retention:  
У вас есть две таблицы:

users: информация о пользователях (уникальный идентификатор и дата регистрации)
user_id
signup_ts

user_activity: события, совершаемые пользователями (идентификатор, дата и тип события)
signup_ts
activity_ts
event_name

Задача: необходимо написать SQL-запрос для расчета когортного удержания (retention rate) помесячно для каждой когорты.

Определения:

Когорта (cohort) — это пользователи, зарегистрировавшиеся в одном и том же календарном месяце (определяется из даты signup_ts в таблице users).
Месяц удержания 0 (month 0) — это месяц регистрации пользователя.
Месяц удержания n (month n) — это календарный месяц, который наступает через n месяцев после месяца регистрации.
Пользователь считается удержанным в месяце n, если у него есть хотя бы одно событие в таблице user_activity в течение этого календарного месяца.
События, произошедшие до регистрации пользователя, игнорируются.

Результат: запрос должен выдать таблицу, содержащую следующие столбцы:

cohort_month — месяц когорты (первый день месяца, например 2025-01-01).
months_since_signup — номер месяца удержания (целое число от 0 до N).
cohort_size — общее количество пользователей в когорте.
retained_users — количество уникальных пользователей из когорты, которые были активны в этот месяц удержания.
retention_rate — процент удержания (retained_users / cohort_size).

In [2]:
import pandas as pd

# 1. Таблица пользователей (6 строк)
# Формируем две когорты: январь 2026 и февраль 2026
users_data = {
    'user_id': [1, 2, 3, 4, 5, 6],
    'signup_ts': [
        '2026-01-15 10:00:00', '2026-01-20 14:30:00', '2026-01-25 09:00:00', # Когорта 2026-01-01 (3 юзера)
        '2026-02-10 11:15:00', '2026-02-15 16:00:00',                         # Когорта 2026-02-01 (2 юзера)
        '2026-03-01 12:00:00'                                                  # Когорта 2026-03-01 (1 юзер)
    ]
}
users = pd.DataFrame(users_data)
users['signup_ts'] = pd.to_datetime(users['signup_ts'])

# 2. Таблица активности пользователей (12 строк во всех списках)
# Наполняем так, чтобы проверить расчет month 0, month 1 и т.д.
user_activity_data = {
    'user_id': [
        1, 1, 1, # Юзер 1 (Январь): активен в январе (month 0), феврале (month 1) и марте (month 2)
        2, 2,    # Юзер 2 (Январь): активен в январе (month 0) и марте (month 2, пропустил месяц 1)
        3,       # Юзер 3 (Январь): системная ошибка (событие ДО регистрации, должно быть проигнорировано)
        4, 4,    # Юзер 4 (Февраль): активен в феврале (month 0) и марте (month 1)
        5,       # Юзер 5 (Февраль): активен только в феврале (month 0)
        1, 6, 2  # Дополнительные действия для разнообразия
    ],
    'activity_ts': [
        '2026-01-15 10:05:00', '2026-02-05 14:00:00', '2026-03-10 11:00:00',
        '2026-01-22 18:00:00', '2026-03-15 09:30:00',
        '2025-12-31 23:59:59', # Событие юзера 3 до его регистрации в январе 2026
        '2026-02-11 10:00:00', '2026-03-01 15:20:00',
        '2026-02-15 16:05:00',
        '2026-01-20 09:00:00', '2026-03-05 14:00:00', '2026-01-22 19:00:00'
    ],
    'event_name': [
        'login', 'view_item', 'purchase',
        'login', 'purchase',
        'click', 
        'login', 'click',
        'login',
        'click', 'login', 'view_item'
    ]
}
user_activity = pd.DataFrame(user_activity_data)
user_activity['activity_ts'] = pd.to_datetime(user_activity['activity_ts'])

# Перемешиваем логи активности для симуляции реальной СУБД
user_activity = user_activity.sample(frac=1, random_state=42).reset_index(drop=True)

print("Датафреймы для когортного анализа успешно созданы!")
print(f"Пользователей в базе: {len(users)}, Записей активности: {len(user_activity)}")


Датафреймы для когортного анализа успешно созданы!
Пользователей в базе: 6, Записей активности: 12


In [3]:
# наилучший вариант SQL
query = """
with t_agg as(
    select 
        user_id,
        date_trunc('month', activity_ts) as month_act
    from user_activity
    group by 
        user_id,
        date_trunc('month', activity_ts)
),
cohort_size as(
    select
        date_trunc('month', signup_ts) as cohort,
        count(*) as cohort_size 
    from users
    group by date_trunc('month', signup_ts)
),
retained as(
    select 
        date_trunc('month', signup_ts) as cohort,
        date_diff('month', date_trunc('month', signup_ts), month_act) as month_passed
    from t_agg t join users u 
        on u.user_id = t.user_id
        and t.month_act >= date_trunc('month', u.signup_ts)
),
retained_agg as(
    select
        cohort,
        month_passed,
        count(*) as retained_users
    from retained
    group by 
        cohort,
        month_passed
)
select 
    cohort,
    cohort_size,
    month_passed,
    retained_users,
    round(100 * retained_users / cohort_size,0) as RR
from retained_agg join cohort_size using(cohort)
order by 
    cohort,
    month_passed
"""
result = duckdb.query(query).to_df()
result

,cohort,cohort_size,month_passed,retained_users,RR
0,2026-01-01,3,0,2,67.0
1,2026-01-01,3,1,1,33.0
2,2026-01-01,3,2,2,67.0
3,2026-02-01,2,0,2,100.0
4,2026-02-01,2,1,1,50.0
5,2026-03-01,1,0,1,100.0


In [4]:
# наилучший вариант python
user_activity['act_month'] = user_activity['activity_ts'].dt.to_period('M')
agg_act = user_activity[['user_id', 'act_month']].drop_duplicates()
users['cohort_month'] = users['signup_ts'].dt.to_period('M')
cohort_size = (
    users.groupby('cohort_month')['user_id']
    .nunique()
    .reset_index(name='cohort_size')
)
df_joined = users.merge(agg_act, on='user_id', how='inner')
df_joined = df_joined[df_joined['act_month'] >= df_joined['cohort_month']]
df_joined['months_since_signup'] = (
    (df_joined['act_month'].dt.year - df_joined['cohort_month'].dt.year) * 12 
    + (df_joined['act_month'].dt.month - df_joined['cohort_month'].dt.month)
)
count_mau = (
    df_joined.groupby(['cohort_month', 'months_since_signup'])['user_id']
    .nunique()
    .reset_index(name='retained_users')
)
result = count_mau.merge(cohort_size, on='cohort_month', how='inner')
result['retention_rate'] = (100 * result['retained_users'] / result['cohort_size']).round()
result.sort_values(['cohort_month', 'months_since_signup']).reset_index(drop=True)

,cohort_month,months_since_signup,retained_users,cohort_size,retention_rate
0,2026-01,0,2,3,67.0
1,2026-01,1,1,3,33.0
2,2026-01,2,2,3,67.0
3,2026-02,0,2,2,100.0
4,2026-02,1,1,2,50.0
5,2026-03,0,1,1,100.0


In [ ]:
df_agg_act = user_activity.copy()
df_agg_act['month_act'] = df_agg_act['activity_ts'].dt.to_period('M')
df_agg_act = df_agg_act[['user_id', 'month_act']].drop_duplicates()
df_retained = df_agg_act.merge(users)
df_retained['cohort'] = df_retained['signup_ts'].dt.to_period('M')
df_retained = df_retained[df_retained['month_act'] >= df_retained['cohort']]
df_retained['month_passed'] = (df_retained['month_act'] - df_retained['cohort']).apply(lambda x: x.n)
df_retained_agg = df_retained[['cohort', 'month_passed']].value_counts().reset_index(name='retained_users')
df_cohort = users.copy()
df_cohort['cohort'] = df_cohort['signup_ts'].dt.to_period('M')
df_cohort_agg = df_cohort['cohort'].value_counts().reset_index(name='cohort_size')
result = df_retained_agg.merge(df_cohort_agg)
result['RR'] = (100 * result['retained_users'] / result['cohort_size']).round()
result.sort_values(['cohort', 'month_passed']).iloc[:,[0, 3, 1, 2, 4]]

In [ ]:
query = """
with agg_act as(   
    select 
        user_id,
        date_trunc('month', activity_ts) as act_month
    from user_activity
    group by user_id, date_trunc('month', activity_ts)
),
count_mau as(
    select
        date_trunc('month', signup_ts) as cohort_month,
        (extract(year from act_month) - extract(year from signup_ts)) * 12 + extract(month from act_month) - extract(month from signup_ts) as months_since_signup,
        count(distinct u.user_id) as retained_users
    from users u join agg_act a 
        on u.user_id = a.user_id
        and act_month >= date_trunc('month', signup_ts)
    group by 
        date_trunc('month', signup_ts),
        ((extract(year from act_month) - extract(year from signup_ts)) * 12 + extract(month from act_month) - extract(month from signup_ts))
),
cohort_size as(
    select 
        date_trunc('month', signup_ts) as cohort_month,
        count(distinct user_id) as cohort_size
    from users
    group by date_trunc('month', signup_ts)
)
select 
    cohort_month,
    months_since_signup,
    cohort_size,
    retained_users,
    round(100 * retained_users / cohort_size) as retention_rate
from count_mau join cohort_size using(cohort_month)
ORDER BY cohort_month, months_since_signup;
"""
result = duckdb.query(query).to_df()
result

Асимметричные курсы валют и мультивалютный финтех  
Контекст
Финтех-платформа проводит транзакции в разных валютах. Нам нужно собрать ежемесячную отчетность по выручке в рублях (RUB) только по реальным (не тестовым) пользователям.

Сложность
Курсы валют обновляются на бирже только по будням (с понедельника по пятницу). Однако пользователи совершают покупки каждый день, включая выходные. Если транзакция совершена в субботу или воскресенье, для неё нужно взять курс валюты за ближайшую предшествующую пятницу.

Таблицы в базе:   
users (Пользователи)
- user_id (int) — идентификатор
- registration_date (date)
- is_test (int) — флаг: 1 (тестовый аккаунт), 0 (реальный пользователь)

transactions (Покупки)
- tx_id (int) — идентификатор платежа
- user_id (int) — кто купил
- amount (numeric) — сумма в локальной валюте
- currency (varchar) — код валюты (например, 'USD', 'EUR')
- tx_date (date) — дата покупки (любой день недели)

currency_rates (Курсы валют)
- currency (varchar) — код валюты
- rate_to_rub (numeric) — курс обмена на рубли
- rate_date (date) — дата фиксации курса (только будние дни)

Задание: напиши запрос, который выведет месяц (в формате 'YYYY-MM' или первого числа месяца) и суммарную выручку в рублях (RUB) за этот месяц, не учитывая тестовых пользователей.

In [2]:
import pandas as pd

# 1. Таблица пользователей (5 строк)
users_data = {
    'user_id': [1, 2, 3, 4, 5],
    'registration_date': ['2026-05-01', '2026-05-02', '2026-05-10', '2026-05-15', '2026-05-20'],
    'is_test': [0, 0, 1, 0, 0] # Юзер 3 — тестовый (его транзакции должны полностью отсеяться)
}
users = pd.DataFrame(users_data)
users['registration_date'] = pd.to_datetime(users['registration_date'])

# 2. Справочник курсов валют (8 строк — только будни, с Пн по Пт)
# 22 мая 2026 — Пятница, 25 мая 2026 — Понедельник
rates_data = {
    'currency': ['USD', 'EUR', 'USD', 'EUR', 'USD', 'EUR', 'USD', 'EUR'],
    'rate_to_rub': [75.0, 85.0, 76.0, 86.0, 75.5, 85.2, 77.0, 87.0],
    'rate_date': [
        '2026-05-20', '2026-05-20', # Среда
        '2026-05-21', '2026-05-21', # Четверг
        '2026-05-22', '2026-05-22', # Пятница (этот курс зафиксируется на все выходные)
        '2026-05-25', '2026-05-25'  # Понедельник
    ]
}
currency_rates = pd.DataFrame(rates_data)
currency_rates['rate_date'] = pd.to_datetime(currency_rates['rate_date'])

# 3. Таблица транзакций (10 строк — любые дни недели)
transactions_data = {
    'tx_id': range(1001, 1011),
    'user_id': [
        1, 1,  # Юзер 1: Будние дни (прямое совпадение курса)
        2, 2,  # Юзер 2: Покупки в Сб и Вс (должны забрать курс за пятницу 22-го мая)
        3,     # Юзер 3: Покупка тестового пользователя (должна отсеяться)
        4, 4,  # Юзер 4: Покупки в понедельник и среду
        5, 5, 5 # Юзер 5: Разные валюты в выходные
    ],
    'amount': [100.0, 50.0, 200.0, 150.0, 500.0, 10.0, 120.0, 300.0, 80.0, 100.0],
    'currency': ['USD', 'EUR', 'USD', 'EUR', 'USD', 'USD', 'EUR', 'USD', 'EUR', 'USD'],
    'tx_date': [
        '2026-05-20', '2026-05-21', # Будни (Ср, Чт)
        '2026-05-23', '2026-05-24', # Выходные (Сб, Вс -> ищут курс от 2026-05-22)
        '2026-05-22',               # Пятница (тестовый юзер, мимо)
        '2026-05-25', '2026-05-20', # Будни (Пн, Ср)
        '2026-05-23', '2026-05-24', # Выходные (Сб, Вс -> ищут курс от 2026-05-22)
        '2026-05-25'                # Будни (Пн)
    ]
}
transactions = pd.DataFrame(transactions_data)
transactions['tx_date'] = pd.to_datetime(transactions['tx_date'])

print("Все мультивалютные датафреймы успешно созданы!")
print(f"Пользователи: {len(users)}, Курсы: {len(currency_rates)}, Транзакции: {len(transactions)}")


Все мультивалютные датафреймы успешно созданы!
Пользователи: 5, Курсы: 8, Транзакции: 10


In [23]:
# более быстрый вариант за счет distinct on
query = """
with t_rate as(
    select distinct on (tx_id)
        date_trunc('month', tx_date) as month,
        amount,
        coalesce(rate_to_rub, 1) as rate
    from transactions t left join currency_rates c
        on t.currency = c.currency
        and tx_date >= rate_date
    where user_id in(select user_id from users where is_test = 0)
    order by 
        tx_id,
        tx_date desc
)
select
    month,
    sum(amount * rate) as sum_transactions
from t_rate
group by month
"""
result = duckdb.query(query).to_df()
result

,month,sum_transactions
0,2026-05-01,87816.0


In [20]:
query = """
with t_rate as(
    select
        date_trunc('month', tx_date) as month,
        amount,
        coalesce(rate_to_rub, 1) as rate,
        row_number() over(partition by tx_id order by tx_date desc) as row_date
    from transactions t left join currency_rates c
        on t.currency = c.currency
        and tx_date >= rate_date
    where user_id in(select user_id from users where is_test = 0)
)
select
    month,
    sum(amount * rate) as sum_transactions
from t_rate
where row_date = 1
group by month
"""
result = duckdb.query(query).to_df()
result

,month,sum_transactions
0,2026-05-01,87816.0


In [8]:
real_users = set(users[users['is_test'] == 0]['user_id'])
t_filtered = transactions[transactions['user_id'].isin(real_users)].copy()
t_filtered = t_filtered.sort_values('tx_date')
rates_sorted = currency_rates.sort_values('rate_date')
t_join = pd.merge_asof(
    t_filtered,
    rates_sorted,
    left_on='tx_date',
    right_on='rate_date',
    by='currency',
    direction='backward'
)
t_join['rate_to_rub'] = t_join['rate_to_rub'].fillna(1)
t_join['amount_rub'] = t_join['amount'] * t_join['rate_to_rub']
t_join['month'] = t_join['tx_date'].dt.to_period('M')
t_join.groupby('month', as_index=False).agg(sum_transactions=('amount_rub', 'sum'))

,month,sum_transactions
0,2026-05,87816.0
